## Bài 2: Dry Bean Dataset - Phân loại các loại hạt đậu

# Mục tiêu
Xây dựng và so sánh hai mô hình phân loại:
1. Logistic Regression
2. K-Nearest Neighbors (KNN)


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
from sklearn.model_selection import cross_val_score, GridSearchCV

# Cấu hình hiển thị
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

np.random.seed(42)

In [4]:
DATA_DIR = Path("Dry_Bean_Dataset")
TRAIN_PATH = DATA_DIR / "dry_bean_train.csv"
TEST_PATH = DATA_DIR / "dry_bean_test.csv"

# Đọc dữ liệu
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

# Kiểm tra dữ liệu
print("\n5 dòng đầu:")
display(train_df.head())

print("\nThông tin train:")
train_df.info()

print("\nPhân bố target trong train:")
print(train_df['class'].value_counts())

Train shape: (10834, 17)
Test shape: (2709, 17)

5 dòng đầu:


,area,perimeter,majoraxislength,minoraxislength,aspectration,eccentricity,convexarea,equivdiameter,extent,solidity,roundness,compactness,shapefactor1,shapefactor2,shapefactor3,shapefactor4,class
0,69471,1069.638,399.100245,225.005782,1.773733,0.825923,71088,297.410868,0.707386,0.977254,0.763027,0.745203,0.005745,0.001093,0.555328,0.985004,CALI
1,82877,1162.581,391.817013,270.836144,1.446694,0.722634,84171,324.841921,0.825986,0.984627,0.770544,0.829065,0.004728,0.001378,0.687349,0.994384,BARBUNYA
2,65042,1023.506,419.202858,198.962774,2.106941,0.880190,65748,287.774298,0.783403,0.989262,0.780231,0.686480,0.006445,0.000883,0.471255,0.992906,HOROZ
3,41315,758.920,287.438268,183.447580,1.566869,0.769858,41704,229.355383,0.791930,0.990672,0.901417,0.797929,0.006957,0.001740,0.636691,0.997611,SIRA
4,91088,1168.645,459.300729,253.950486,1.808623,0.833243,91799,340.553731,0.789051,0.992255,0.838119,0.741461,0.005042,0.000940,0.549765,0.994318,CALI



Thông tin train:
<class 'pandas.DataFrame'>
RangeIndex: 10834 entries, 0 to 10833
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   area             10834 non-null  int64  
 1   perimeter        10834 non-null  float64
 2   majoraxislength  10834 non-null  float64
 3   minoraxislength  10834 non-null  float64
 4   aspectration     10834 non-null  float64
 5   eccentricity     10834 non-null  float64
 6   convexarea       10834 non-null  int64  
 7   equivdiameter    10834 non-null  float64
 8   extent           10834 non-null  float64
 9   solidity         10834 non-null  float64
 10  roundness        10834 non-null  float64
 11  compactness      10834 non-null  float64
 12  shapefactor1     10834 non-null  float64
 13  shapefactor2     10834 non-null  float64
 14  shapefactor3     10834 non-null  float64
 15  shapefactor4     10834 non-null  float64
 16  class            10834 non-null  str    
dtypes: fl

In [5]:
target = 'class'

# Tách feature và target
X_train = train_df.drop(columns=[target])
y_train = train_df[target].copy()

X_test = test_df.drop(columns=[target])
y_test = test_df[target].copy()

# Kiểm tra
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

# Danh sách features
features = X_train.columns.tolist()
print(f"\nFeatures: {features}")
print(f"Số lượng features: {len(features)}")

# Kiểm tra target distribution
print("\nPhân bố target trong train:")
print(y_train.value_counts().sort_index())
print("\nPhân bố target trong test:")
print(y_test.value_counts().sort_index())

X_train shape: (10834, 16)
X_test shape: (2709, 16)
y_train shape: (10834,)
y_test shape: (2709,)

Features: ['area', 'perimeter', 'majoraxislength', 'minoraxislength', 'aspectration', 'eccentricity', 'convexarea', 'equivdiameter', 'extent', 'solidity', 'roundness', 'compactness', 'shapefactor1', 'shapefactor2', 'shapefactor3', 'shapefactor4']
Số lượng features: 16

Phân bố target trong train:
class
BARBUNYA    1057
BOMBAY       418
CALI        1304
DERMASON    2837
HOROZ       1488
SEKER       1621
SIRA        2109
Name: count, dtype: int64

Phân bố target trong test:
class
BARBUNYA    265
BOMBAY      104
CALI        326
DERMASON    709
HOROZ       372
SEKER       406
SIRA        527
Name: count, dtype: int64


- Logistic Regression: hội tụ nhanh hơn
- KNN: khoảng cách Euclidean bị ảnh hưởng bởi scale

In [6]:
scaler = StandardScaler()

# Fit scaler trên train data và transform
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data
X_test_scaled = scaler.transform(X_test)

print("Mean of scaled train features (gần 0):", X_train_scaled.mean(axis=0).round(6))
print("Std of scaled train features (gần 1):", X_train_scaled.std(axis=0).round(6))

# Chuyển về DataFrame
X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=features
)
X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=features
)

print("\nSau scaling:")
display(X_train_scaled_df.head())

Mean of scaled train features (gần 0): [-0.  0. -0. -0.  0.  0.  0. -0. -0.  0. -0.  0.  0. -0.  0. -0.]
Std of scaled train features (gần 1): [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]

Sau scaling:


,area,perimeter,majoraxislength,minoraxislength,aspectration,eccentricity,convexarea,equivdiameter,extent,solidity,roundness,compactness,shapefactor1,shapefactor2,shapefactor3,shapefactor4
0,0.560509,1.002462,0.925291,0.503525,0.786671,0.823696,0.582245,0.749885,-0.871715,-2.122169,-1.873432,-0.898136,-0.724272,-1.052294,-0.903132,-2.305622
1,1.017202,1.435815,0.840324,1.520932,-0.547166,-0.300388,1.021247,1.212690,1.562425,-0.541524,-1.746374,0.466374,-1.625335,-0.574239,0.435162,-0.156327
2,0.409630,0.787368,1.159812,-0.074614,2.145667,1.414271,0.403061,0.587300,0.688437,0.452249,-1.582650,-1.853622,-0.103932,-1.404477,-1.755383,-0.495135
3,-0.398662,-0.446283,-0.377380,-0.419042,-0.057031,0.213544,-0.403740,-0.398319,0.863458,0.754599,0.465657,-0.040240,0.349745,0.032920,-0.078360,0.583062
4,1.296921,1.464089,1.627602,1.146080,0.928970,0.903357,1.277206,1.477773,0.804356,1.093852,-0.604207,-0.959023,-1.346560,-1.308562,-0.959527,-0.171608


- Xây dựng và đánh giá Logistic Regression

In [ ]:
lr_model = LogisticRegression(
    solver='lbfgs',
    max_iter=1000,
    random_state=42
)

lr_model.fit(X_train_scaled, y_train)

#Dự đoán
y_pred_lr = lr_model.predict(X_test_scaled)
y_pred_proba_lr = lr_model.predict_proba(X_test_scaled)

print("\nKết quả Logistic Regression:")

accuracy_lr = accuracy_score(y_test, y_pred_lr)
print(f"Accuracy: {accuracy_lr:.4f}")

# Tính các metric
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))

# Macro và weighted averages
precision_macro_lr = precision_score(y_test, y_pred_lr, average='macro')
recall_macro_lr = recall_score(y_test, y_pred_lr, average='macro')
f1_macro_lr = f1_score(y_test, y_pred_lr, average='macro')

precision_weighted_lr = precision_score(y_test, y_pred_lr, average='weighted')
recall_weighted_lr = recall_score(y_test, y_pred_lr, average='weighted')
f1_weighted_lr = f1_score(y_test, y_pred_lr, average='weighted')

print(f"\nMacro average:")
print(f"  Precision: {precision_macro_lr:.4f}")
print(f"  Recall: {recall_macro_lr:.4f}")
print(f"  F1-score: {f1_macro_lr:.4f}")

print(f"\nWeighted average:")
print(f"  Precision: {precision_weighted_lr:.4f}")
print(f"  Recall: {recall_weighted_lr:.4f}")
print(f"  F1-score: {f1_weighted_lr:.4f}")

#Cross-validation trên train set
cv_scores_lr = cross_val_score(
    lr_model,
    X_train_scaled,
    y_train,
    cv=5,
    scoring='accuracy'
)
print(f"\nCross-validation accuracy (5-fold):")
print(f"  Mean: {cv_scores_lr.mean():.4f} (+/- {cv_scores_lr.std():.4f})")
print(f"  Scores: {cv_scores_lr}")


Kết quả Logistic Regression:
Accuracy: 0.9192

Classification Report:
              precision    recall  f1-score   support

    BARBUNYA       0.93      0.89      0.91       265
      BOMBAY       1.00      1.00      1.00       104
        CALI       0.91      0.94      0.93       326
    DERMASON       0.93      0.91      0.92       709
       HOROZ       0.96      0.94      0.95       372
       SEKER       0.93      0.94      0.94       406
        SIRA       0.86      0.88      0.87       527

    accuracy                           0.92      2709
   macro avg       0.93      0.93      0.93      2709
weighted avg       0.92      0.92      0.92      2709


Macro average:
  Precision: 0.9307
  Recall: 0.9300
  F1-score: 0.9302

Weighted average:
  Precision: 0.9197
  Recall: 0.9192
  F1-score: 0.9193

Cross-validation accuracy (5-fold):
  Mean: 0.9239 (+/- 0.0024)
  Scores: [0.9247808  0.92293493 0.92108906 0.92247347 0.92797784]


- Xây dựng và đánh giá KNN

In [ ]:
print("\nTìm K optimal bằng GridSearch...")

knn_param_grid = {
    'n_neighbors': range(1, 31),
    'weights': ['uniform', 'distance'],
    'p': [1, 2]  # 1: Manhattan, 2: Euclidean
}

knn_gs = GridSearchCV(
    KNeighborsClassifier(),
    knn_param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

knn_gs.fit(X_train_scaled, y_train)

print(f"\nBest parameters: {knn_gs.best_params_}")
print(f"Best CV score: {knn_gs.best_score_:.4f}")

# Train KNN với best parameters
knn_best = knn_gs.best_estimator_

#Dự đoán trên test
y_pred_knn = knn_best.predict(X_test_scaled)
y_pred_proba_knn = knn_best.predict_proba(X_test_scaled)

#Đánh giá
print("\nKết quả KNN:")

accuracy_knn = accuracy_score(y_test, y_pred_knn)
print(f"Accuracy: {accuracy_knn:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_knn))

# Macro và weighted averages
precision_macro_knn = precision_score(y_test, y_pred_knn, average='macro')
recall_macro_knn = recall_score(y_test, y_pred_knn, average='macro')
f1_macro_knn = f1_score(y_test, y_pred_knn, average='macro')

precision_weighted_knn = precision_score(y_test, y_pred_knn, average='weighted')
recall_weighted_knn = recall_score(y_test, y_pred_knn, average='weighted')
f1_weighted_knn = f1_score(y_test, y_pred_knn, average='weighted')

print(f"\nMacro average:")
print(f"  Precision: {precision_macro_knn:.4f}")
print(f"  Recall: {recall_macro_knn:.4f}")
print(f"  F1-score: {f1_macro_knn:.4f}")

print(f"\nWeighted average:")
print(f"  Precision: {precision_weighted_knn:.4f}")
print(f"  Recall: {recall_weighted_knn:.4f}")
print(f"  F1-score: {f1_weighted_knn:.4f}")

#Thử nghiệm với các K khác nhau.
k_values = range(1, 31)
knn_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, weights='distance', p=2)
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=5)
    knn_scores.append(scores.mean())

print(f"\nBest K: {k_values[np.argmax(knn_scores)]} với accuracy: {max(knn_scores):.4f}")


Tìm K optimal bằng GridSearch...
Fitting 5 folds for each of 120 candidates, totalling 600 fits

Best parameters: {'n_neighbors': 18, 'p': 2, 'weights': 'distance'}
Best CV score: 0.9270

Kết quả KNN:
Accuracy: 0.9188

Classification Report:
              precision    recall  f1-score   support

    BARBUNYA       0.95      0.88      0.92       265
      BOMBAY       1.00      1.00      1.00       104
        CALI       0.91      0.96      0.94       326
    DERMASON       0.92      0.91      0.91       709
       HOROZ       0.96      0.93      0.95       372
       SEKER       0.93      0.94      0.94       406
        SIRA       0.85      0.89      0.87       527

    accuracy                           0.92      2709
   macro avg       0.93      0.93      0.93      2709
weighted avg       0.92      0.92      0.92      2709


Macro average:
  Precision: 0.9332
  Recall: 0.9297
  F1-score: 0.9310

Weighted average:
  Precision: 0.9199
  Recall: 0.9188
  F1-score: 0.9190

Best K: 18 v

- So sánh giữa hai mô hình

In [ ]:
comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision (macro)', 'Recall (macro)', 'F1 (macro)',
               'Precision (weighted)', 'Recall (weighted)', 'F1 (weighted)'],
    'Logistic Regression': [
        accuracy_lr,
        precision_macro_lr,
        recall_macro_lr,
        f1_macro_lr,
        precision_weighted_lr,
        recall_weighted_lr,
        f1_weighted_lr
    ],
    'KNN': [
        accuracy_knn,
        precision_macro_knn,
        recall_macro_knn,
        f1_macro_knn,
        precision_weighted_knn,
        recall_weighted_knn,
        f1_weighted_knn
    ]
})

comparison['Logistic Regression'] = comparison['Logistic Regression'].round(4)
comparison['KNN'] = comparison['KNN'].round(4)

comparison['Difference'] = comparison['KNN'] - comparison['Logistic Regression']
comparison['Difference'] = comparison['Difference'].round(4)

display(comparison)

best_accuracy = max(accuracy_lr, accuracy_knn)
best_model = "Logistic Regression" if accuracy_lr >= accuracy_knn else "KNN"
print(f"\nMô hình có accuracy hơn nhất: {best_model} ({best_accuracy:.4f})")

,Metric,Logistic Regression,KNN,Difference
0,Accuracy,0.9192,0.9188,-0.0004
1,Precision (macro),0.9307,0.9332,0.0025
2,Recall (macro),0.9300,0.9297,-0.0003
3,F1 (macro),0.9302,0.9310,0.0008
4,Precision (weighted),0.9197,0.9199,0.0002
5,Recall (weighted),0.9192,0.9188,-0.0004
6,F1 (weighted),0.9193,0.9190,-0.0003



Mô hình có accuracy hơn nhất: Logistic Regression (0.9192)
